# Employee Attrition Risk Analysis
**Dataset:** IBM HR Analytics Employee Attrition (Kaggle)  
**Analyst:** Om S Omanwar  
**Purpose:** Identify high-risk attrition segments and frame business-ready intervention recommendations  

---

## Notebook Structure
1. Setup and Data Load
2. Exploratory Data Analysis (EDA)
3. Preprocessing
4. Modelling (RandomForest + Class Imbalance Handling)
5. Feature Importance
6. High-Risk Cohort Profile
7. Business Brief Summary

---
## 1. Setup and Data Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder

# Plot styling
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.family'] = 'sans-serif'
sns.set_palette('Blues_r')

# Load dataset
# Download from: https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset
df = pd.read_csv('WA_Fn-UseC_-HR-Employee-Attrition.csv')
print(f'Shape: {df.shape}')
df.head(3)

---
## 2. Exploratory Data Analysis

**Goal:** Understand the data before modelling. Key questions:
- What is the overall attrition rate?
- Are there obvious distributional differences between leavers and stayers?
- Where is class imbalance, and how severe?

In [ ]:
# Null check
print('=== Null counts ===')
print(df.isnull().sum().sum(), 'total nulls')

# Attrition split
attrition_counts = df['Attrition'].value_counts()
attrition_rate = attrition_counts['Yes'] / len(df) * 100
print(f'\nAttrition rate: {attrition_rate:.1f}%')
print(attrition_counts)

# Business interpretation:
# 16.1% attrition vs 10-12% industry benchmark = elevated risk.
# Also signals class imbalance: ~5.2 non-leavers per leaver.
# Default classifiers will exploit this by predicting 'Stay' almost always.

In [ ]:
# Attrition rate by department
dept_attr = df.groupby('Department')['Attrition'].apply(lambda x: (x == 'Yes').mean() * 100).reset_index()
dept_attr.columns = ['Department', 'AttritionRate']

fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Bar: dept attrition
ax[0].barh(dept_attr['Department'], dept_attr['AttritionRate'], color=['#0D1B4B', '#2563EB', '#93C5FD'])
ax[0].set_xlabel('Attrition Rate (%)')
ax[0].set_title('Attrition Rate by Department')
ax[0].axvline(attrition_rate, color='red', linestyle='--', label=f'Overall avg ({attrition_rate:.1f}%)')
ax[0].legend()

# Histogram: Monthly income by attrition
ax[1].hist(df[df['Attrition'] == 'No']['MonthlyIncome'], bins=30, alpha=0.6, label='Stay', color='#0D1B4B')
ax[1].hist(df[df['Attrition'] == 'Yes']['MonthlyIncome'], bins=30, alpha=0.7, label='Leave', color='#EF4444')
ax[1].set_xlabel('Monthly Income (USD)')
ax[1].set_title('Monthly Income Distribution by Attrition')
ax[1].legend()

plt.tight_layout()
plt.show()

# Business interpretation:
# Leavers are concentrated in the lower income band.
# Sales department shows above-average attrition -- likely overtime + commission pressure.

In [ ]:
# Overtime analysis
ot_attr = df.groupby('OverTime')['Attrition'].apply(lambda x: (x == 'Yes').mean() * 100).reset_index()
ot_attr.columns = ['OverTime', 'AttritionRate']
print('Attrition rate by OverTime status:')
print(ot_attr.to_string(index=False))

# Age distribution
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df[df['Attrition'] == 'No']['Age'], bins=20, alpha=0.6, label='Stay', color='#0D1B4B')
ax.hist(df[df['Attrition'] == 'Yes']['Age'], bins=20, alpha=0.7, label='Leave', color='#EF4444')
ax.set_xlabel('Age')
ax.set_title('Age Distribution by Attrition Status')
ax.legend()
plt.tight_layout()
plt.show()

# Business interpretation:
# Employees flagged for overtime leave at ~2.8x the rate of non-overtime employees.
# This is a controllable variable -- workload redistribution is an addressable lever.
# Leavers skew younger: concentration at 25-32.

In [ ]:
# Correlation heatmap (numeric features only)
numeric_df = df.select_dtypes(include=[np.number]).copy()
numeric_df['Attrition_bin'] = (df['Attrition'] == 'Yes').astype(int)

top_corr = numeric_df.corr()['Attrition_bin'].drop('Attrition_bin').abs().sort_values(ascending=False).head(10)
print('Top 10 numeric features correlated with attrition:')
print(top_corr.round(3).to_string())

---
## 3. Preprocessing

Steps:
- Binary encode: Attrition, OverTime, Gender
- One-hot encode: Department, JobRole, MaritalStatus, EducationField, BusinessTravel
- Drop constant columns: EmployeeCount, StandardHours, Over18 (no variance)
- Drop ID column: EmployeeNumber (not a predictor)

In [ ]:
df_model = df.copy()

# Binary encodings
df_model['Attrition'] = (df_model['Attrition'] == 'Yes').astype(int)
df_model['OverTime'] = (df_model['OverTime'] == 'Yes').astype(int)
df_model['Gender'] = (df_model['Gender'] == 'Male').astype(int)

# Drop constant + ID columns
drop_cols = ['EmployeeCount', 'StandardHours', 'Over18', 'EmployeeNumber']
df_model.drop(columns=drop_cols, inplace=True)

# One-hot encode remaining categoricals
cat_cols = ['Department', 'JobRole', 'MaritalStatus', 'EducationField', 'BusinessTravel']
df_model = pd.get_dummies(df_model, columns=cat_cols, drop_first=True)

print(f'Final feature matrix shape: {df_model.shape}')
print(f'Attrition class balance: {df_model["Attrition"].value_counts().to_dict()}')

---
## 4. Modelling

### Why RandomForest?
- Handles non-linear relationships and feature interactions (e.g. low income AND overtime is more than additive)
- Built-in feature importance scores
- Robust to correlated features without regularisation tuning

### Class Imbalance Strategy
Default classifiers on 16% positive class will optimise for accuracy by predicting 'Stay' almost always -- achieving ~84% accuracy but near-zero recall on the attrition class. This is operationally useless: we want to identify who will leave, not confirm who will stay.

Two approaches tested:
1. `class_weight='balanced'`: adjusts sample weights so the minority class contributes proportionally more to the loss
2. Threshold tuning: lowering the decision cutoff from 0.5 to 0.15 identifies more predicted leavers at the cost of more false positives

In [ ]:
# Train/test split
X = df_model.drop(columns=['Attrition'])
y = df_model['Attrition']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Baseline model (no imbalance correction)
rf_baseline = RandomForestClassifier(n_estimators=100, random_state=42)
rf_baseline.fit(X_train, y_train)
y_pred_baseline = rf_baseline.predict(X_test)

print('=== BASELINE MODEL ===')
print(classification_report(y_test, y_pred_baseline, target_names=['Stay', 'Leave']))

# Interpretation: notice that recall on the 'Leave' class is near-zero.
# The model is essentially predicting 'Stay' for almost everyone.
# This is the class imbalance problem in action.

In [ ]:
# Improved model: class_weight='balanced' + threshold tuning
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)

# Get probabilities for threshold tuning
y_proba = rf.predict_proba(X_test)[:, 1]

# Threshold tuning: 0.15 trades precision for recall
# Rationale: cost of missing a flight risk > cost of a false positive intervention
THRESHOLD = 0.15
y_pred_tuned = (y_proba >= THRESHOLD).astype(int)

print(f'=== TUNED MODEL (threshold = {THRESHOLD}) ===')
print(classification_report(y_test, y_pred_tuned, target_names=['Stay', 'Leave']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_tuned)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Stay', 'Leave'])
fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix (threshold=0.15)')
plt.tight_layout()
plt.show()

---
## 5. Feature Importance

Feature importance from RandomForest reflects how much each variable reduces impurity across all decision trees. Useful for identifying the levers management can actually pull.

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
top_features = importances.head(15)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#0D1B4B' if i < 5 else '#93C5FD' for i in range(len(top_features))]
top_features.plot(kind='barh', ax=ax, color=colors[::-1])
ax.set_xlabel('Feature Importance Score')
ax.set_title('Top 15 Features Predicting Attrition')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print('Top 5 features:')
print(top_features.head().round(4).to_string())

# Business interpretation:
# MonthlyIncome, OverTime, Age are the consistent top 3.
# These are actionable: compensation review, workload audit, and career development programmes
# for early-career employees map directly to each driver.

---
## 6. High-Risk Cohort Profile

Using the rules learned from feature importance + EDA to define the high-risk segment.

In [ ]:
# High-risk cohort: rules based on top drivers
hr_mask = (
    (df['Age'] < 35) &
    (df['JobLevel'].isin([1, 2])) &
    (df['MonthlyIncome'] < 5000) &
    (df['OverTime'] == 'Yes') &
    (df['YearsAtCompany'] < 3)
)

high_risk = df[hr_mask].copy()
print(f'High-risk cohort size: {len(high_risk)} employees ({len(high_risk)/len(df)*100:.1f}% of workforce)')
print(f'Attrition rate in cohort: {(high_risk["Attrition"] == "Yes").mean()*100:.1f}%')
print(f'Overall attrition rate: {(df["Attrition"] == "Yes").mean()*100:.1f}%')

# Department breakdown
print('\nDepartment breakdown of high-risk cohort:')
print(high_risk['Department'].value_counts().to_string())

In [ ]:
# Add model probability scores to full dataset and filter cohort
df_scored = df_model.copy()
df_scored['attrition_prob'] = rf.predict_proba(X)[:, 1]

cohort_scores = df_scored[hr_mask]['attrition_prob']
print(f'Avg attrition probability score (high-risk cohort): {cohort_scores.mean():.3f}')
print(f'Avg attrition probability score (full population): {df_scored["attrition_prob"].mean():.3f}')
print(f'Ratio: {cohort_scores.mean() / df_scored["attrition_prob"].mean():.1f}x higher')

---
## 7. Business Brief Summary

This is the deliverable that matters. Any data person can run a model. Almost none of them can write a business case from it.

In [ ]:
# ROI calculation
avg_monthly_income = df[df['Attrition'] == 'Yes']['MonthlyIncome'].mean()
avg_annual_income = avg_monthly_income * 12
replacement_cost_per_employee = avg_annual_income * 1.5  # conservative: 1.5x annual salary

cohort_size = len(high_risk)
estimated_leavers = int(cohort_size * (high_risk['Attrition'] == 'Yes').mean())
total_replacement_cost = estimated_leavers * replacement_cost_per_employee

intervention_cost = 4300000  # Rs 43L total
employees_retained_estimate = 110
saving_per_retained = replacement_cost_per_employee
gross_savings = employees_retained_estimate * saving_per_retained
net_savings = gross_savings - intervention_cost
roi = gross_savings / intervention_cost

print('=== BUSINESS CASE SUMMARY ===')
print(f'High-risk cohort: {cohort_size} employees')
print(f'Avg replacement cost per employee: Rs {replacement_cost_per_employee:,.0f}')
print(f'Intervention cost (3 levers): Rs {intervention_cost:,.0f}')
print(f'Estimated employees retained: {employees_retained_estimate}')
print(f'Gross savings: Rs {gross_savings:,.0f}')
print(f'Net savings: Rs {net_savings:,.0f}')
print(f'Implied ROI: {roi:.1f}x')

print()
print('=== PITCH TO LEADERSHIP ===')
print(f'''
This model identifies that approximately {cohort_size} employees are at elevated attrition risk in the next 6-12 months.
Estimated replacement cost if no action taken: Rs {total_replacement_cost/10000000:.1f}Cr.

Three targeted interventions (compensation review, overtime audit, stock option expansion)
are estimated at Rs 43L and projected to retain ~110 employees, saving Rs {net_savings/10000000:.1f}Cr net.

Implied ROI: {roi:.0f}x.

Recommendation: Prioritise the compensation review for the high-risk cohort in Q1.
This is the single highest-leverage action and does not require structural change.
''')